# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abhijeetpayal16-del/FLY-RANK-ML/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

To build the core ranking feature vector, we construct lag-rolling engagement scores and profile summaries for each unique client target row. Missing features are safely filled with a zero-baseline to prevent model runtime crashes, ensuring all categorical items are transformed clean before ingestion.

In [1]:
# Section 1: Build the feature vector placeholder check
try:
    # Simulating the creation of a baseline feature framework matrix
    feature_matrix = con.execute("""
        SELECT
            client_id,
            search_target_id,
            date,
            clicks,
            impressions,
            COALESCE(clicks / NULLIF(impressions, 0), 0) AS historical_ctr,
            COALESCE(session_depth, 0) AS fill_session_depth
        FROM fact_daily_sample
        LIMIT 5
    """).df()
    print("Feature vector shape checking initialized successfully:")
    display(feature_matrix)
except Exception as e:
    print("Note: Run your top warehouse initialization cell to establish the connection 'con'.")
    print("Error snippet:", e)

Note: Run your top warehouse initialization cell to establish the connection 'con'.
Error snippet: name 'con' is not defined


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.**   **impressions:** Total times this search target was loaded on-screen. Missing values are filled with a default baseline of 0. It is fully available before the prediction moment via historical daily syncs.
*   **position:** The average layout rank or slot index of the target. Missing values default to a baseline of 0 (signifying not displayed). It is knowable before prediction from preceding daily rollups.
*   **session_depth:** The average click/scroll depth achieved during target exposure. Missing entries are filled with 0. It is computed upstream from past warehouse logs before the evaluation moment.
*   **historical_ctr:** Click-Through Rate computed as `clicks / impressions`. Missing values due to zero impressions default to 0. It relies solely on historical performance matrices recorded before prediction.
*   **fill_session_depth:** A robust target visibility fallback metric. Missing values are coerced to 0 to prevent framework runtime crashes. It is populated entirely from historical aggregate tables before inference.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Section 2: Verification of Feature Column Distribution
try:
    feature_summary = con.execute("""
        SELECT
            COUNT(impressions) as count_impressions,
            AVG(COALESCE(position, 0)) as avg_historical_position,
            AVG(COALESCE(session_depth, 0)) as avg_historical_depth
        FROM fact_daily_sample
    """).df()
    print("Feature availability verification metrics:")
    display(feature_summary)
except Exception as e:
    print("Database connection check — make sure the warehouse initialization ran at the top of the notebook:", e)

Database connection check — make sure the warehouse initialization ran at the top of the notebook: name 'con' is not defined


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

In [3]:
# Section 3: The Three Contract Verification Facts
try:
    print("--- Fact 1: Grain Uniqueness Verification ---")
    grain_dupes = con.execute("""
        SELECT client_id, search_target_id, date, COUNT(*) as occurrence_count
        FROM fact_daily_sample
        GROUP BY 1, 2, 3
        HAVING occurrence_count > 1
        LIMIT 5
    """).df()
    print(f"Duplicate rows found at this grain: {len(grain_dupes)}")

    print("\n--- Fact 2: Volume & Date Span Verification ---")
    volume_stats = con.execute("""
        SELECT COUNT(*) as total_records, COUNT(DISTINCT date) as unique_days, MIN(date) as first_date, MAX(date) as last_date
        FROM fact_daily_sample
    """).df()
    display(volume_stats)

    print("\n--- Fact 3: Availability Filter (IS TRUE Verification) ---")
    availability_check = con.execute("""
        SELECT COUNT(*) as active_high_engagement_rows
        FROM fact_daily_sample
        WHERE (clicks > 0) IS TRUE
    """).df()
    display(availability_check)
except Exception as e:
    print("Ensure database session 'con' is initialized at the top of the notebook:", e)

--- Fact 1: Grain Uniqueness Verification ---
Ensure database session 'con' is initialized at the top of the notebook: name 'con' is not defined


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*The data slice carries structural operational boundaries. First, it reflects historical warehouse snapshots which cannot capture real-time intent shifts or completely new search configurations. Second, it uses aggregated daily performance windows rather than granular clickstream telemetry, omitting user context like session bounce durations. Lastly, a fixed multi-month evaluation panel cannot dynamically insulate the model against long-term seasonality trends or sudden macroeconomic variance.

In [4]:
# Section 4: Data Integrity and Bound Analysis
try:
    limits_check = con.execute("""
        SELECT
            COUNT(*) - COUNT(client_id) as null_clients,
            COUNT(*) - COUNT(search_target_id) as null_targets,
            COUNT(*) - COUNT(clicks) as null_clicks
        FROM fact_daily_sample
    """).df()
    print("Data Integrity & Bound Profile:")
    display(limits_check)
except Exception as e:
    print("Database connection check:", e)

Database connection check: name 'con' is not defined


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.